In [ ]:
import scanpy as sc
import pandas as pd
import os
import matplotlib.pyplot as plt

############ QC for GSE273062
############################
data_dir = "/home/kongfh/konglab/scDATA/GSE273062/RAW_Data"
sample_mapping = {
    'SuHx_1': 'GSM8420502_SuHx_1',
    'SuHx_2': 'GSM8420503_SuHx_2',
    'SuHx_3': 'GSM8420504_SuHx_3',
    'VeNx_1': 'GSM8420505_VeNx_1',
    'VeNx_2': 'GSM8420506_VeNx_2'
}
adatas = []
for sample_name, prefix in sample_mapping.items():
    print(f"In progress: {sample_name} ...")
    mtx_path = os.path.join(data_dir, f"{prefix}_matrix.mtx.gz")
    features_path = os.path.join(data_dir, f"{prefix}_features.tsv.gz")
    barcodes_path = os.path.join(data_dir, f"{prefix}_barcodes.tsv.gz")
    adata = sc.read_mtx(mtx_path).T 
    genes = pd.read_csv(features_path, header=None, sep='\t')
    barcodes = pd.read_csv(barcodes_path, header=None, sep='\t')
    adata.var_names = genes[1].values  
    adata.obs_names = barcodes[0].values
    adata.var_names_make_unique()
    adata.obs['sample'] = sample_name
    group_name = sample_name.split('_')[0]
    adata.obs['group'] = group_name
    adatas.append(adata)
print("Integrating all samples...")
adata_combined = sc.concat(adatas, join='outer', label='sample', keys=sample_mapping.keys(), index_unique='-')
adata_combined.write_h5ad("/home/kongfh/konglab/scDATA/GSE273062/adata_raw.h5ad")

adata.var["mt"] = adata.var_names.str.startswith("mt")
adata.var["ribo"] = adata.var_names.str.startswith(("Rps","Rpl"))
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt","ribo"], percent_top=None, log1p=False, inplace=True)
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', "pct_counts_ribo"], jitter=0.4, multi_panel=True)

adata = adata[adata.obs.pct_counts_mt < 7, : ]
adata = adata[adata.obs.pct_counts_ribo < 30, : ]
adata = adata[adata.obs.n_genes_by_counts < 6000, : ]
adata = adata[adata.obs.n_genes_by_counts > 500, : ]

adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)
adata.layers['data'] = adata.X.copy()
sc.pp.highly_variable_genes(adata, n_top_genes = 2000, batch_key="sample")
adata.raw = adata
adata = adata[:, adata.var["highly_variable"]]
sc.pp.scale(adata, max_value=10)
adata.layers['scaled'] = adata.X.copy()

sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True)
import scanpy.external as sce
import harmonypy as hm
sce.pp.harmony_integrate(adata, 'sample')
sc.pp.neighbors(adata, n_pcs = 50,  use_rep='X_pca_harmony', random_state=10)
adata.obsm["X_pca"] = adata.obsm["X_pca_harmony"]
sc.tl.umap(adata, random_state=10)
sc.tl.leiden(adata, resolution=0.4)
sc.pl.umap(adata, color=['group','leiden'], legend_loc='on data')
adata = adata[adata.obs['cell_type'] != 'Proliferating macrophages (pMacro)'].copy() ###Extremely low cell count, potentially indicative of contamination
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
markers = sc.get.rank_genes_groups_df(adata, None)
markers = markers[(markers.pvals_adj < 0.05) & (markers.logfoldchanges > 0.1)]
markers.to_excel("/home/kongfh/konglab/scDATA/GSE273062/markers.xlsx", index = False)
markers = pd.read_excel("/home/kongfh/konglab/scDATA/GSE273062/markers.xlsx")
cell_type = {"0":"Alveolar macrophaghe (aendo)","1":"Non-classical monocytes (ncMono)","2":"B cells (B)","3":"CD8 T cells (T_cd8)",
             "4":"Interstitial endophages (iendo)","5":"Mesothelial cells (Meso)","6":"Fibroblasts (Fb)","7":"Endothelial cells 1 (EC)",
             "8":"NK cells (NK)","9":"CIliated cells (Ciliated)","10":"Naive T cells (T_naive)","11":"Club cells (Club)",
            "12":"Alveolar epithelial (AT)","13":"Conventional dendritic (cDC)","14":"Neutrophils (PMN)","15":"Smooth muscle cells (SMC)",
            "16":"Smooth muscle cells (SMC)","17":"Classical monocytes (cMono)","18":"unkown","19":"Endothelial cells 2(EC)",
            "20":"Club cells (Club)","21":"Mast cells (Mast)","22":"Plasmacytoid dendritic (pDC)","23":"Unknown",
            "24":"Unknown","25":"Unknown","26":"Unknown","27":"Unknown"
             }
adata.obs["cell_type"] = adata.obs["leiden"].map(cell_type)
markers["cell_type"] = markers["group"].map(cell_type)
markers.to_excel("/home/kongfh/konglab/scDATA/GSE273062/markers.xlsx", index = False)
adata.write_h5ad("/home/kongfh/konglab/scDATA/GSE273062/adata_annotation.h5ad")

In [ ]:
adata = sc.read_h5ad("adata_annotation.h5ad")
###Fig 1G
sc.pl.umap(adata, color=['cell_type','group'], wspace=0.9, show=False)
plt.savefig("/home/kongfh/konglab/scDATA/GSE273062/All_UMAP.png", dpi=600, bbox_inches='tight')
###Fig 1H
my_color = LinearSegmentedColormap.from_list("my_color", color_list)
fig, ax = plt.subplots(figsize=(7.5, 7))
sc.pl.umap(
    adata,
    color=['Txndc5'],
    wspace=0.2,
    show=False,
    color_map=my_color,
    size=30,
    ax=ax
)
plt.savefig(
    "/home/kongfh/konglab/scDATA/GSE273062/Txndc5_UMAP.png",
    dpi=600,
    bbox_inches='tight'
)
###Fig 1I
import matplotlib.pyplot as plt
gene = "Txndc5"

use_raw = adata.raw is not None

df = sc.get.obs_df(
    adata,
    keys=["cell_type", gene],
    use_raw=use_raw,
)
order = (
    df.groupby("cell_type", observed=True)[gene]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(16, 5))

sc.pl.violin(
    adata,
    keys=gene,
    groupby="cell_type",
    order=order,
    use_raw=use_raw,
    stripplot=False,
    dodge=False,
    rotation=90,
    ax=ax,
    show=False,
)

plt.tight_layout()
fig.savefig(f"{gene}_violin.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
## Extraction of endothelial cell subpopulations
adata_endo = adata.raw.to_adata()
adata_endo = adata_endo[adata.obs['cell_type'] == 'ECs '].copy()
celltype_map = {
    "1": "Gas-exchange homeostatic ECs",
    "2": "Remodeling-primed transitional ECs",
    "3": "TXNDC5-high ECM-producing ECs"
}
adata_endo.obs["celltype"] = (
    adata_endo.obs["leiden_a"]
    .map(celltype_map))
adata_endo.write_h5ad("/home/kongfh/konglab/scDATA/GSE273062/adata_endo.h5ad")

In [ ]:
adata_endo = sc.read_h5ad("adata_endo.h5ad")
###Fig 6A
ax2 = sc.pl.umap(
    adata_endo,
    color='group',
    size=30,
    show=False
)
###Fig 6B
ax = sc.pl.umap(
    adata_endo,
    color='celltype',
    size=30,
    show=False
)
plt.show()
###Fig 6C
genes = [
    'Ca4', 'Cldn5', 'Apln', 'Kdr', 'Selenop',
    'Myo10', 'Itga1', 'Col4a1', 'Fabp4', 'Adamts9',
    'Txndc5', 'Fn1', 'Vcan', 'Bgn', 'Ccn2'
]

dp = sc.pl.dotplot(
    adata_endo,
    var_names=genes,
    groupby='celltype',
    categories_order=cell_order,
    standard_scale='var',
    cmap='RdBu_r',
    dot_max=0.8,
    dot_min=0.05,
    smallest_dot=5,
    largest_dot=200,
    figsize=(10, 4),
    dendrogram=False,
    use_raw=False,
    return_fig=True
)

dp.make_figure()

ax = dp.get_axes()['mainplot_ax']

ax.axvline(4.5, color='black', linestyle=':', linewidth=0.8)
ax.axvline(8.5, color='black', linestyle=':', linewidth=0.8)

plt.show()
###Fig 6D
sc.pl.umap(
    adata,
    color=['Txndc5'],
    wspace=0.2,
    show=False,
    color_map=my_color,
    size=30,
    ax=ax
)
###Fig S9A
sc.pl.violin(
    adata_endo,
    keys="TXNDC5",
    groupby="celltype",
    stripplot=False,
    jitter=False,
    rotation=0,
    palette=["#D9D9D9", "#6FA0F5", "#EF6262"],
    ylabel="UMI counts of TXNDC5",
    title="GSE273062"
)

### Fig S9B (Data for go enrichment)
sc.tl.rank_genes_groups(
    adata_endo,
    groupby="celltype",
    groups=["TXNDC5-high ECM-producing ECs"],
    reference="rest",
    method="wilcoxon",
    use_raw=False,
    pts=True
)
marker_genes = sc.get.rank_genes_groups_df(
    adata_endo,
    group="TXNDC5-high ECM-producing ECs"
)
###Fig S10C (left GSE273062)
##A gene-scoring strategy was employed, with the selection of individual genes guided by the Methods section of the article.
gene_set = ["Egln3", "Vegfa", "Serpine1", "Ldha", "Edn1"]
genes_present = [g for g in gene_set if g in adata_endo_new.var_names]
print("Genes found:", genes_present)
sc.tl.score_genes(
    adata_endo_new,
    gene_list=genes_present,
    score_name="Epas1_activity_score",
    use_raw=False
)
import seaborn as sns
plt.figure(figsize=(12, 5))
sns.violinplot(
    data=adata_endo_new.obs,
    x="celltype",
    y="Epas1_activity_score",
    inner="box",
    palette="Set2",
    cut=0
)
plt.xticks(rotation=45, ha="right")
plt.xlabel("Cell type")
plt.ylabel("Epas1_activity_score")
plt.title("Epas1_activity_scoreEndothelial Cell Types")
plt.tight_layout()
plt.show()

df_score = adata_endo_new.obs[
    ["celltype", "Epas1_activity_score"]
].copy()
df_score.to_csv(
    "Epas1_activity_score_by_celltype.csv",
    index=True,
    index_label="cell_id"
)
print("CSV exported successfully!")
###Fig S11C
umap = adata_endo.obsm["X_umap"]
tx = adata_endo[:, "TXNDC5"].X.toarray().flatten()
bgn = adata_endo[:, "BGN"].X.toarray().flatten()

fig, ax = plt.subplots(1, 3, figsize=(12, 4))

for a in ax:
    a.scatter(*umap.T, c="lightgrey", s=3)

ax[0].scatter(*umap[tx>0].T, c="red", s=3)
ax[0].set_title("TXNDC5", color="red")

ax[1].scatter(*umap[bgn>0].T, c="lime", s=3)
ax[1].set_title("BGN", color="limegreen")

ax[2].scatter(*umap[(tx>0)&(bgn==0)].T, c="red", s=3)
ax[2].scatter(*umap[(bgn>0)&(tx==0)].T, c="lime", s=3)
ax[2].scatter(*umap[(tx>0)&(bgn>0)].T, c="orange", s=3)
ax[2].set_title("Merge")

plt.tight_layout()
plt.show()
###

In [ ]:
#### In-depth characterization of endothelial cells based on the six established canonical lung endothelial subclusters.
adata_endo = adata.raw.to_adata()
adata_endo = adata_endo[adata.obs['cell_type'] == 'ECs '].copy()
sc.pp.highly_variable_genes(adata_endo, n_top_genes=2000)
adata_endo.raw = adata_endo
adata_endo = adata_endo[:, adata_endo.var.highly_variable]
sc.tl.pca(adata_endo, svd_solver="arpack", random_state=10)
sc.pl.pca_variance_ratio(adata_endo, log=True, n_pcs = 50)
import scanpy.external as sce
import harmonypy as hm
sce.pp.harmony_integrate(adata_endo, 'sample')
adata_endo.obsm["X_pca"] = adata_endo.obsm["X_pca_harmony"]
sc.pp.neighbors(adata_endo, n_pcs=15, random_state=10)
sc.tl.umap(adata_endo, random_state=10)
sc.tl.leiden(adata_endo, resolution=0.2, random_state=10)
write_h5ad(adata_endo, "/home/kongfh/konglab/scDATA/GSE273062/adata_endo_new.h5ad")

In [ ]:
adata_endo = sc.read_h5ad("adata_endo_new.h5ad")
#### Identification of cluster-specific marker genes
print(adata_endo.obs.columns)
print(adata_endo.obs["leiden_r02"].value_counts())
sc.tl.rank_genes_groups(
    adata_endo,
    groupby="leiden_r02",      
    method="wilcoxon",
    use_raw=False
)
sc.pl.rank_genes_groups(
    adata_endo,
    n_genes=20,
    sharey=False
)

#### Cell annotation

celltype_map = {
    '0': 'Lymphatic ECs',
    '1': 'Pulmonary venous ECs',
    '2': 'Systemic venous ECs',
    '3': 'gCaps',
    '4': 'Pulmonary arterial ECs',
    '5': 'Lymphatic ECs',
    '6': 'gCaps',
    '7': 'aCaps'
}
adata_endo.obs['celltype_a'] = (
    adata_endo.obs['leiden_r02']
    .astype(str)
    .map(celltype_map)
)
###Fig 6E
sc.pl.umap(
    adata_endo,
    color=['leiden_r02', 'celltype_a'],
    legend_loc='right margin',
    wspace=0.4,
    frameon=False
)
plt.savefig(
    'adata_endo_celltype_umap.png',
    dpi=600,
    bbox_inches='tight'
)

###Fig 6G
genes = [
    "Txndc5", "Bgn", "Vcan", "Ccn2", "Fn1", "Foxc2"]
custom_cmap = LinearSegmentedColormap.from_list(
    "custom_blue_gold_red",
    ["white", "#F2C14E", "#F56F70"]
)
sc.pl.matrixplot(
    adata_endo,
    var_names=genes,
    groupby="celltype_a",
    standard_scale="var",  
    cmap=custom_cmap,
    figsize=(4.7, 3),
    show=False
)
plt.savefig("Endo_marker_matrixplot.png",dpi=1200,bbox_inches="tight")
plt.show()

###Fig S9C,D
genes_found = [
    "Txndc5", "Bgn", "Vcan", "Ccn2", "Fn1",
    "Dkk2", "Sox17", "Ednrb",
    "Apln", "Aplnr", "Plvap", "Cpe",
    "Nr2f2", "Col15a1", "Vwa1", "Flt4", "Reln"
]
custom_cmap = mcolors.LinearSegmentedColormap.from_list(
    "smooth_gray_red",
    [
        "lightgray",
        "#F59A9F",
        "#F77E82",
        "#F56F70"
    ]
)
ncols = 4
nrows = math.ceil(len(genes_found) / ncols)
fig = sc.pl.umap(
    adata_endo,
    color=genes_found,
    cmap=custom_cmap,
    size=20,
    frameon=False,
    ncols=ncols,
    wspace=0.35,
    show=False,
    return_fig=True
)
fig.set_size_inches(16, nrows * 4)
for ax in fig.axes:
    if ax.get_title() in genes_found:
        ax.set_box_aspect(1)
        ax.set_aspect('equal', adjustable='datalim')
        
fig.savefig(
    "Endo_marker_genes_UMAP.png",
    dpi=1200,
    bbox_inches="tight"
)
plt.show()

In [ ]:
### Fig 6F ###
###The correlation between the newly and previously annotated cell populations was assessed using the OE method.
## Data preprocessing
adata_endo_new = sc.read_h5ad("adata_endo_new.h5ad")
adata_endo = sc.read_h5ad("adata_endo.h5ad")

adata_endo_new.obs["celltype_new"] = (
    adata_endo.obs["celltype"]
    .reindex(adata_endo_new.obs_names)
)
adata_endo_new.obs["celltype_new1"] = (
    adata_endo_new.obs["celltype_a"].astype(str)
)

mask = (
    adata_endo_new.obs["celltype"]
    == "TXNDC5-high ECM-producing ECs"
)

expr = adata_endo_new[mask, "Txndc5"].X

if hasattr(expr, "toarray"):
    expr = expr.toarray()

expr = np.asarray(expr).flatten()

adata_endo_new.obs.loc[mask, "celltype_new1"] = np.where(
    expr > 0,
    "Txndc5+ ECM-producing ECs",
    "Txndc5- ECM-producing ECs"
)


In [ ]:
### Fig 6F ###
## Data  Calculation 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

###==========================================
### 1. Table construction
x_order = [
    "Gas-exchange homeostatic ECs",
    "Remodeling-primed transitional ECs",
    "Txndc5+ ECM-producing ECs",
]

obs = adata_endo_new.obs[
    ["celltype_new", "celltype_new1"]
].dropna()

ct = pd.crosstab(
    obs["celltype_new"],    
    obs["celltype_new1"],   
)

ct = ct.loc[
    ct.sum(axis=1) > 0,
    ct.sum(axis=0) > 0
]

if ct.empty:
    raise ValueError("Cells lacking paired valid old and new annotations.")

missing = [name for name in x_order if name not in ct.columns]
if missing:
    raise ValueError(f"The following newly annotated cell types were not detected in the dataset:{missing}")

###==========================================
### 2. Calculation of Expected Counts and Observed-to-Expected (O/E) Ratios
row_total = ct.sum(axis=1) 
col_total = ct.sum(axis=0)  
total = ct.to_numpy().sum()

expected = pd.DataFrame(
    np.outer(row_total.to_numpy(), col_total.to_numpy()) / total,
    index=ct.index,
    columns=ct.columns,
)
# O/E = Observed Count / Expected Count (based on background abundance)
oe = ct / expected
with np.errstate(divide="ignore"):
    log2_oe = np.log2(oe)
oe_show = oe.loc[:, x_order]
log2_oe_show = log2_oe.loc[:, x_order]
plot_mat = log2_oe_show.clip(lower=-3, upper=3)

###==========================================
### 3. Row-wise Z-score normalization of the final displayed matrix

raw_mat = log2_oe.loc[:, x_order].T.copy()
finite_mat = raw_mat.replace([np.inf, -np.inf], np.nan)
row_mean = finite_mat.mean(axis=1)
row_sd = finite_mat.std(axis=1, ddof=1)
safe_sd = row_sd.where(
    row_sd.gt(0) & np.isfinite(row_sd),
    1.0
)
row_z = (
    finite_mat
    .sub(row_mean, axis="index")
    .div(safe_sd, axis="index")
)
row_z.to_csv("Row_Zscore_log2OE_matrix.csv")

### ==========================================
### 4. Heatmap visualization: Both color intensity and numerical values represent row-wise Z-scores.
enrichment_cmap = LinearSegmentedColormap.from_list(
    "white_yellow_pink",
    ["white", "#F2C14E", "#F56F70"]
)
enrichment_cmap.set_bad(enrichment_cmap(0.0))
z_lim = 1.5 
fig, ax = plt.subplots(figsize=(11, 5.8))
sns.heatmap(
    row_z,
    cmap=enrichment_cmap,
    center=0,
    vmin=-z_lim,
    vmax=z_lim,
    annot=row_z.to_numpy(),
    fmt=".2f",
    mask=row_z.isna(),
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Row Z-score of log2(O/E)"},
    ax=ax
)

ax.set_xlabel("Old cell type")
ax.set_ylabel("New cell type")
ax.set_title("Row-standardized mapping\nNumbers: row Z-score")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color("black")
    spine.set_linewidth(1.2)
for i, j in np.argwhere(row_z.isna().to_numpy()):
    ax.text(
        j + 0.5, i + 0.5,
        "-1.5",
        ha="center",
        va="center",
        color="black"
    )
fig.tight_layout()
fig.savefig(
    "Background_adjusted_OE_row_Zscore_heatmap.png",
    dpi=1200,
    bbox_inches="tight"
)

plt.show()
plt.close(fig)